# **Suggestion Algorithm**
The goal of this section is to implement a simple algorithm that, based on the genre chosen by the user and based on his taste, finds the user with most similar taste and suggests movies based on what the most similar user has liked.

In [1]:
import pandas as pd

In [ ]:
ratings = pd.read_csv('/Users/beatricecitterio/ratings.csv')
movies = pd.read_csv('/Users/beatricecitterio/movies.csv')
# change paths with where you stored the data

In [3]:
ratings = ratings.drop(columns='timestamp')

In [4]:
movie_counts = ratings['movieId'].value_counts().reset_index()
movie_counts.columns = ['movieId', 'count']

In [6]:
movie_counts # create this df so that we know the popularity of each movie (i.e. how many times it has been rated)

,movieId,count
0,318,102929
1,356,100296
2,296,98409
3,2571,93808
4,593,90330
...,...,...
84427,288825,1
84428,288467,1
84429,287221,1
84430,284087,1


In [7]:
genres = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'Any Genre']

In [8]:
top_movies = {}
for genre in genres:
    if genre == 'Any Genre':
        mov = movies.merge(movie_counts, on = 'movieId').sort_values(by= 'count', ascending = False)
        top_movies[genre] = mov.nlargest(20, 'count')
    else: 
        genre_movies = movies[movies['genres'].str.contains(genre, case=False)]
        genre_movies = genre_movies.merge(movie_counts, on = 'movieId').sort_values(by= 'count', ascending = False)
        top_movies[genre] = genre_movies.nlargest(20, 'count')


This dictionary stores the 20 most popular (i.e. most rated) movies for each genre.

In [9]:
top_movies['Any Genre']

,movieId,title,genres,count
314,318,"Shawshank Redemption, The (1994)",Crime|Drama,102929
351,356,Forrest Gump (1994),Comedy|Drama|Romance|War,100296
292,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,98409
2480,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808
585,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,90330
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,85010
2867,2959,Fight Club (1999),Action|Crime|Drama|Thriller,77332
475,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233
522,527,Schindler's List (1993),Drama|War,73849
4888,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,73122


In [10]:
top_movies['Thriller']

,movieId,title,genres,count
52,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,98409
431,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808
110,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,90330
487,2959,Fight Club (1999),Action|Crime|Drama|Thriller,77332
89,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233
9,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,67750
8,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,63298
111,608,Fargo (1996),Comedy|Crime|Drama|Thriller,58031
2456,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,57931
134,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,57224


Now, the user is asked to fill out a form in order to understand his taste. First, he needs to specify the genre he is interested in (among those proposed by us). Suppose this is given, we will call it 'genre' and it will be a string. He will also be asked how long he wants the survey to be (5, 10 or 20 movies to rate). Given these, the algorithm will output the list of movies he needs to rate.

In [33]:
def movies_to_rate(genre: str, n = 10):
    return list(top_movies[genre][:n].title)

Given this list of movies, the user will give a rating from 0 to 5 to each one of them. If he hasn't seen the movie, he must put 'Not seen'. Suppose this rating is given as an array, called new_rating, of size n. Optionally, one can choose also the number of suggestions to receive.

In [34]:
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np

In [40]:
def suggestion(new_rating: list, genre: str, n = 10, number_of_suggestions = 3):
    new_rating_dict = {}
    for i in range(n):
        new_rating_dict[list(top_movies[genre][:n].movieId)[i]] = new_rating[i]

    new_rating_filtered =  {k: v for k, v in new_rating_dict.items() if v != 'Not seen'}
    filtered_ratings = ratings[ratings['movieId'].isin(new_rating_filtered.keys())]

    pivot_df = filtered_ratings.pivot(index='userId', columns='movieId', values='rating').fillna(2.5)

    new_user_ratings = pd.DataFrame([new_rating_filtered], index=['new_user'])
    dissimilarities = euclidean_distances(pivot_df, new_user_ratings)

    most_similar_user = pivot_df.index[np.argmin(dissimilarities)]

    print(f'Most similar user ID: {most_similar_user}')

    user_ratings = ratings[ratings['userId'] == most_similar_user]
    movies_by_genre = movies[movies['genres'].str.contains(genre, case=False)]
    user_ratings = user_ratings[user_ratings['movieId'].isin(movies_by_genre.movieId)]

    user_ratings_filtered = user_ratings.sort_values(by = 'rating', ascending=False)

    suggested_movies = user_ratings_filtered[~user_ratings_filtered['movieId'].isin(new_rating_filtered.keys())]
    suggested_movies = suggested_movies.merge(movie_counts).sort_values(by = ['rating', 'count'], ascending=False)
    final_suggestions = suggested_movies[:number_of_suggestions]
    final_suggestions = movies[movies['movieId'].isin(final_suggestions.movieId)].title
    return final_suggestions


## **NEW EXAMPLE**

In [41]:
genre = 'Children'
n = 20

In [42]:
movies_to_rate(genre, n)

['Toy Story (1995)',
 'Shrek (2001)',
 'Lion King, The (1994)',
 'Aladdin (1992)',
 'Finding Nemo (2003)',
 'Monsters, Inc. (2001)',
 'Incredibles, The (2004)',
 'Beauty and the Beast (1991)',
 'E.T. the Extra-Terrestrial (1982)',
 'WALL·E (2008)',
 'Up (2009)',
 'Babe (1995)',
 "Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)",
 'Home Alone (1990)',
 'Toy Story 2 (1999)',
 'Willy Wonka & the Chocolate Factory (1971)',
 'Wizard of Oz, The (1939)',
 'Jumanji (1995)',
 'Ratatouille (2007)',
 "Bug's Life, A (1998)"]

In [43]:
new_rating = [
    5,
    5,
    4.5,
    'Not seen',
    5,
    4.5,
    5,
    4,
    4,
    'Not seen',
    4,
    'Not seen',
    5,
    4.5,
    5,
    5,
    'Not seen',
    'Not seen',
    4.5,
    'Not seen'
]

In [45]:
suggestion(new_rating, genre, n=20, number_of_suggestions=5)

Most similar user ID: 111252


580                    Aladdin (1992)
898          Wizard of Oz, The (1939)
932      It's a Wonderful Life (1946)
14815              Toy Story 3 (2010)
29990               Inside Out (2015)
Name: title, dtype: object